In [2]:
from KEGG.pathway.consumer import consume_pathway_ids
from local_db.SQL.SQLCaller import SQL_ETL

sql_caller = SQL_ETL()
pathway_ids = consume_pathway_ids(sql_caller)

SQLStateManager is not ready (database connection failed)
SQLStateManager is not ready (database connection failed)
missing_tables []


In [ ]:
from local_db.SQL.SQLCaller import SQL_ETL

uuid = '5B146EBC-98C5-4242-A049-A5331A641CB4'
sql_caller = SQL_ETL(run_id=uuid)
sql_caller.sql_state.drop_table('PathwayKGMLMeta', schema="dbo")

TypeError: SQL_ETL.__init__() got an unexpected keyword argument 'run_id'

In [3]:
pathway_ids

['hsa00010',
 'hsa00020',
 'hsa00030',
 'hsa00040',
 'hsa00051',
 'hsa00052',
 'hsa00053',
 'hsa00061',
 'hsa00062',
 'hsa00071',
 'hsa00100',
 'hsa00120',
 'hsa00130',
 'hsa00140',
 'hsa00190',
 'hsa00220',
 'hsa00230',
 'hsa00232',
 'hsa00240',
 'hsa00250',
 'hsa00260',
 'hsa00270',
 'hsa00280',
 'hsa00290',
 'hsa00310',
 'hsa00330',
 'hsa00340',
 'hsa00350',
 'hsa00360',
 'hsa00380',
 'hsa00400',
 'hsa00410',
 'hsa00430',
 'hsa00440',
 'hsa00450',
 'hsa00470',
 'hsa00480',
 'hsa00500',
 'hsa00510',
 'hsa00511',
 'hsa00512',
 'hsa00513',
 'hsa00514',
 'hsa00515',
 'hsa00520',
 'hsa00524',
 'hsa00531',
 'hsa00532',
 'hsa00533',
 'hsa00534',
 'hsa00541',
 'hsa00561',
 'hsa00562',
 'hsa00563',
 'hsa00564',
 'hsa00565',
 'hsa00590',
 'hsa00591',
 'hsa00592',
 'hsa00600',
 'hsa00601',
 'hsa00603',
 'hsa00604',
 'hsa00620',
 'hsa00630',
 'hsa00640',
 'hsa00650',
 'hsa00660',
 'hsa00670',
 'hsa00730',
 'hsa00740',
 'hsa00750',
 'hsa00760',
 'hsa00770',
 'hsa00780',
 'hsa00785',
 'hsa00790',

In [5]:
from pprint import pprint

query = sql_caller._build_merge_query('PathwayKGMLMeta')

pprint(query)

('\n'
 '        MERGE INTO dbo.PathwayKGMLMeta AS target\n'
 '        USING staging.PathwayKGMLMeta AS source\n'
 '        ON target.pathway_id = source.pathway_id\n'
 '        WHEN MATCHED AND target.kgml_hash <> source.kgml_hash THEN UPDATE '
 'SET target.kgml_hash = source.kgml_hash, target.last_checked = '
 'source.last_checked\n'
 '        WHEN NOT MATCHED THEN INSERT (pathway_id, kgml_hash, last_checked) '
 'VALUES (source.pathway_id, source.kgml_hash, source.last_checked)\n'
 "        OUTPUT 'd402af46-c1a4-4857-a2a4-c1a151df47c1' AS run_id, $action AS "
 'action,\n'
 '                    COALESCE(inserted.pathway_id, deleted.pathway_id) AS '
 'pathway_id\n'
 '                    INTO diff.PathwayKGMLMeta (run_id, action, pathway_id);\n'
 '        ',
 True)


In [1]:
from neo4j import GraphDatabase, Driver
import os
from utils.colored_text import RED, GREEN, RESET, YELLOW
from local_db.SQLCaller import SQL_ETL
from typing import Any

GRAPH_URI = os.getenv("NEO4J_URI")
GRAPH_AUTH = (os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD"))

graph_driver = GraphDatabase.driver(GRAPH_URI, auth=GRAPH_AUTH)

python-dotenv could not parse statement starting at line 2


In [1]:
from local_db.SQLCaller import SQL_ETL

sql_caller = SQL_ETL()

sql_caller.sql_state.drop_table_with_constraints("reaction_participants", schema="dbo")

python-dotenv could not parse statement starting at line 2


Successfully connected to the database!
missing_tables []
Successfully connected to the database!
Dropping table dbo.reaction_participants


In [5]:
import requests

UNIPROT_STREAM_URL = "https://rest.uniprot.org/uniprotkb/stream"

def fetch_function_annotations( uniprot_accessions: list[str], batch_size: int = 200) -> list[dict]:
    """
    Pulls function annotation (and a few useful adjacent fields) for a list
    of UniProt accessions via the stream endpoint, in TSV form.
    """
    fields = "accession,cc_function"
    results = []

    for i in range(0, len(uniprot_accessions), batch_size):
        batch = uniprot_accessions[i:i + batch_size]
        query = " OR ".join(f"accession:{acc}" for acc in batch)

        resp = requests.get(
            UNIPROT_STREAM_URL,
            params={"query": query, "fields": fields, "format": "tsv"},
            timeout=60,
        )
        resp.raise_for_status()

        lines = resp.text.strip().split("\n")
        header = lines[0].split("\t")
        for line in lines[1:]:
            row = dict(zip(header, line.split("\t")))
            results.append(row)

    return results

fetch_function_annotations(["P01308", "Q9H2U9"])

[{'Entry': 'P01308',
  'Function [CC]': 'FUNCTION: Insulin decreases blood glucose concentration. It increases cell permeability to monosaccharides, amino acids and fatty acids. It accelerates glycolysis, the pentose phosphate cycle, and glycogen synthesis in liver.'},
 {'Entry': 'Q9H2U9',
  'Function [CC]': 'FUNCTION: Required for normal male fertility via maintenance of epithelial cell morphology in the caput epididymis and subsequently correct epididymis lumen structure required for sperm development (By similarity). Plays a role in sperm motility, flagella morphology and tyrosine phosphorylation during sperm capacitance (By similarity). Plays a role in normal expression levels of HSPA5, ITM2B and ADAM2 in sperm both prior to and post-capacitation (By similarity). This is a non catalytic metalloprotease-like protein (By similarity). {ECO:0000250|UniProtKB:O35227}.'}]

In [2]:
from KEGG.KEGGCaller import KEGGCaller
from KEGG.pathway.producer import parse_kegg_kgml
from local_db.SQLCaller import SQL_ETL

kegg_caller = KEGGCaller()
sql_caller = SQL_ETL()

entity_list, entity_path_mem_list, intrx_list, rx_list, rx_pt_list= parse_kegg_kgml(kegg_caller, 'hsa00010')

Successfully connected to the database!
missing_tables ['reaction_participants']
Missing production tables: ['reaction_participants']
Table dbo.reaction_participants created successfully.
Successfully connected to the database!


In [6]:
rxprt_id = set(rx_pt.reaction_id for rx_pt in rx_pt_list)
rx_id = set(rx.reaction_id for rx in rx_list)

print(rxprt_id, rx_id)

{'R03270', 'R00341', 'R02187', 'R09532', 'R00014', 'R00200', 'R01788', 'R01015', 'R05134', 'R00703', 'R02189', 'R01516', 'R01061', 'R01600', 'R01602', 'R00658', 'R02569', 'R05133', 'R00762', 'R00947', 'R00229', 'R04394', 'R09085', 'R05198', 'R05132', 'R10860', 'R01196', 'R01786', 'R00710', 'R01512', 'R01662', 'R00199', 'R00726', 'R13199', 'R09479', 'R02738', 'R00746', 'R00756', 'R00754', 'R07618', 'R09127', 'R01058', 'R00206', 'R00959', 'R00431', 'R00764', 'R02739', 'R00755', 'R01068', 'R09086', 'R07159', 'R05805', 'R01063', 'R00711', 'R00235', 'R01518'} {'R03270', 'R00341', 'R02187', 'R09532', 'R00014', 'R00200', 'R01788', 'R01015', 'R05134', 'R00703', 'R02189', 'R01516', 'R01061', 'R01600', 'R01602', 'R00658', 'R02569', 'R05133', 'R00762', 'R00947', 'R00229', 'R04394', 'R09085', 'R05198', 'R05132', 'R10860', 'R01196', 'R01786', 'R00710', 'R01512', 'R01662', 'R00199', 'R00726', 'R13199', 'R09479', 'R02738', 'R00746', 'R00756', 'R00754', 'R07618', 'R09127', 'R01058', 'R00206', 'R00959'

In [1]:
from KEGG.KEGGCaller import KEGGCaller
from KEGG.pathway.producer import produce_kegg_kgml, produce_kgml_structure
from local_db.SQLCaller import SQL_ETL

kegg_caller = KEGGCaller()
sql_caller = SQL_ETL()


pathway_id = 'hsa00010'
produce_kgml_structure(kegg_caller, sql_caller, pathway_id)
produce_kegg_kgml(kegg_caller, sql_caller, pathway_id)

python-dotenv could not parse statement starting at line 2


Successfully connected to the database!
missing_tables ['reaction_participants']
Missing production tables: ['reaction_participants']
Table dbo.reaction_participants created successfully.
Successfully connected to the database!
missing_tables ['reaction_participants']
missing_tables ['reaction_participants']
Table staging.reaction_participants created successfully.

                INSERT INTO staging.entities (entity_id, entity_type)
                SELECT ?, ?
                WHERE NOT EXISTS (
                    SELECT 1 FROM staging.entities t
                    WHERE t.entity_id = ?
                );
                
Chunk 1: Inserted 159 rows into staging.entities (duplicates skipped).
Ensured diff table exists: diff.entities
Changes detected for entities!
missing_tables []

                INSERT INTO staging.EntityPathMem (pathway_id, entity_id)
                SELECT ?, ?
                
Chunk 1: Inserted 159 rows into staging.EntityPathMem (duplicates skipped).
 Computed 

In [2]:
from local_db.SQLCaller import SQL_ETL

sql_caller = SQL_ETL()
sql_caller.stage_and_upsert(target_table='reactions', data=rx_list)

Successfully connected to the database!
missing_tables []
Successfully connected to the database!
missing_tables []
Staging Data into reactions

                INSERT INTO staging.reactions (reaction_id, reaction_type, pathway_id)
                SELECT ?, ?, ?
                WHERE NOT EXISTS (
                    SELECT 1 FROM staging.reactions t
                    WHERE t.reaction_id = ?
                );
                 [('R01068', 'reversible', 'hsa00010', 'R01068'), ('R00710', 'reversible', 'hsa00010', 'R00710'), ('R00711', 'reversible', 'hsa00010', 'R00711'), ('R00746', 'reversible', 'hsa00010', 'R00746'), ('R00754', 'reversible', 'hsa00010', 'R00754'), ('R00014', 'irreversible', 'hsa00010', 'R00014'), ('R03270', 'irreversible', 'hsa00010', 'R03270'), ('R02569', 'reversible', 'hsa00010', 'R02569'), ('R00703', 'reversible', 'hsa00010', 'R00703'), ('R00200', 'irreversible', 'hsa00010', 'R00200'), ('R00658', 'reversible', 'hsa00010', 'R00658'), ('R01518', 'reversible', 'hsa0001

False

In [4]:
sql_caller.stage_and_upsert(target_table='reaction_participants', data=rx_pt_list)

missing_tables []
Staging Data into reaction_participants

                INSERT INTO staging.reaction_participants (reaction_id, entity_id, role)
                SELECT ?, ?, ?
                WHERE NOT EXISTS (
                    SELECT 1 FROM staging.reaction_participants t
                    WHERE t.reaction_id = ?
                );
                 [('R01068', 'C00354', 'substrate', 'R01068'), ('R01068', 'C00118', 'product', 'R01068'), ('R01068', 'C00111', 'product', 'R01068'), ('R01068', 'C00354', 'substrate', 'R01068'), ('R01068', 'C00118', 'product', 'R01068'), ('R01068', 'C00111', 'product', 'R01068'), ('R01068', 'C00354', 'substrate', 'R01068'), ('R01068', 'C00118', 'product', 'R01068'), ('R01068', 'C00111', 'product', 'R01068'), ('R00710', 'C00084', 'substrate', 'R00710'), ('R00710', 'C00033', 'product', 'R00710'), ('R00710', 'C00084', 'substrate', 'R00710'), ('R00710', 'C00033', 'product', 'R00710'), ('R00710', 'C00084', 'substrate', 'R00710'), ('R00710', 'C00033', 'pro

In [2]:
reaction_ids = set()
for reaction in rx_pt_list:
    reaction_ids.add(reaction.reaction_id)

In [3]:
len(reaction_ids)

56

In [5]:
kegg_caller.build_entity_tables(rx_list)

AttributeError: 'Reaction' object has no attribute 'type'

In [6]:
#rx_pt_list # Reaction participants and their role. This compound is a substrate in this irreversible reaction.
rx_list

[Reaction(reaction_id='R01068', name=None, definition=None, comment=None, reaction_type='reversible', pathway_id='hsa00010'),
 Reaction(reaction_id='R00710', name=None, definition=None, comment=None, reaction_type='reversible', pathway_id='hsa00010'),
 Reaction(reaction_id='R00229', name=None, definition=None, comment=None, reaction_type=None, pathway_id='hsa00010'),
 Reaction(reaction_id='R00711', name=None, definition=None, comment=None, reaction_type='reversible', pathway_id='hsa00010'),
 Reaction(reaction_id='R00755', name=None, definition=None, comment=None, reaction_type=None, pathway_id='hsa00010'),
 Reaction(reaction_id='R00746', name=None, definition=None, comment=None, reaction_type='reversible', pathway_id='hsa00010'),
 Reaction(reaction_id='R00754', name=None, definition=None, comment=None, reaction_type='reversible', pathway_id='hsa00010'),
 Reaction(reaction_id='R00014', name=None, definition=None, comment=None, reaction_type=None, pathway_id='hsa00010'),
 Reaction(reacti

In [4]:
rx_ids = set(rx.reaction_id for rx in rx_list)
reaction_ids - rx_ids

set()

In [6]:
len(rx_list)

56

In [1]:
from local_db.GOCaller import GOCaller

go_caller = GOCaller()
go_caller.fetch_latest_go_json()
graph = go_caller.read_to_graph()

go-basic.json unchanged, skipping.


In [42]:
graph['nodes']

[{'id': 'http://purl.obolibrary.org/obo/GO_0000001',
  'lbl': 'mitochondrion inheritance',
  'type': 'CLASS',
  'meta': {'definition': {'val': 'The distribution of mitochondria, including the mitochondrial genome, into daughter cells after mitosis or meiosis, mediated by interactions between mitochondria and the cytoskeleton.',
    'xrefs': ['GOC:mcc', 'PMID:10873824', 'PMID:11389764']},
   'synonyms': [{'pred': 'hasExactSynonym',
     'val': 'mitochondrial inheritance'}],
   'basicPropertyValues': [{'pred': 'http://www.geneontology.org/formats/oboInOwl#hasOBONamespace',
     'val': 'biological_process'}]}},
 {'id': 'http://purl.obolibrary.org/obo/GO_0000002',
  'lbl': 'obsolete mitochondrial genome maintenance',
  'type': 'CLASS',
  'meta': {'definition': {'val': 'OBSOLETE. The maintenance of the structure and integrity of the mitochondrial genome; includes replication and segregation of the mitochondrial chromosome.',
    'xrefs': ['GOC:ai', 'GOC:vw']},
   'comments': ['This term des

In [47]:
[property for node in graph['nodes'] if 'meta' in node if 'basicPropertyValues' in node['meta'] 
 for property in node['meta']['basicPropertyValues'] if not '#' in property['pred']]

[{'pred': 'http://purl.obolibrary.org/obo/IAO_0100001',
  'val': 'http://purl.obolibrary.org/obo/GO_0022414'},
 {'pred': 'http://purl.obolibrary.org/obo/IAO_0000231',
  'val': 'http://purl.obolibrary.org/obo/IAO_0000227'},
 {'pred': 'http://purl.obolibrary.org/obo/IAO_0100001',
  'val': 'http://purl.obolibrary.org/obo/GO_0008150'},
 {'pred': 'http://purl.obolibrary.org/obo/IAO_0000231',
  'val': 'http://purl.obolibrary.org/obo/IAO_0000227'},
 {'pred': 'http://purl.obolibrary.org/obo/IAO_0100001',
  'val': 'http://purl.obolibrary.org/obo/GO_0000008'},
 {'pred': 'http://purl.obolibrary.org/obo/IAO_0000231',
  'val': 'http://purl.obolibrary.org/obo/IAO_0000227'},
 {'pred': 'http://purl.obolibrary.org/obo/IAO_0100001',
  'val': 'http://purl.obolibrary.org/obo/GO_0007131'},
 {'pred': 'http://purl.obolibrary.org/obo/IAO_0000231',
  'val': 'http://purl.obolibrary.org/obo/IAO_0000227'},
 {'pred': 'http://purl.obolibrary.org/obo/IAO_0100001',
  'val': 'http://purl.obolibrary.org/obo/GO_0006450'

In [21]:
def flatten_go_props(node):
    if node.get("lbl") is None:
        print(node)

    flat = {"name": node.get("lbl")}
    metadata = {k: v for k, v in node.get('meta', {}).items() if v is not None}
    if not metadata:
        print(node)

    if "definition" in metadata:
        flat["definition"] = metadata["definition"]["val"]
        flat["definition_xref"] = metadata["definition"].get("xrefs", [])
    if "synonyms" in metadata:
        flat["synonyms"] = [s["val"] for s in metadata["synonyms"]]
    if "basicPropertyValues" in metadata:
        for bpv in metadata['basicPropertyValues']:
            key = bpv["pred"].split("#")[-1]
            flat[key] = bpv["val"]
    
    flat['deprecated'] = metadata.get('deprecated', False)

    return flat


payload = []
for node in graph["nodes"]:
    if node['type'] == 'CLASS' and node.get("lbl") is not None:
        id = node['id'].split('/')[-1]
        payload.append({"id": id, "props" : flatten_go_props(node)})
    
payload

[{'id': 'GO_0000001',
  'props': {'name': 'mitochondrion inheritance',
   'definition': 'The distribution of mitochondria, including the mitochondrial genome, into daughter cells after mitosis or meiosis, mediated by interactions between mitochondria and the cytoskeleton.',
   'definition_xref': ['GOC:mcc', 'PMID:10873824', 'PMID:11389764'],
   'synonyms': ['mitochondrial inheritance'],
   'hasOBONamespace': 'biological_process',
   'deprecated': False}},
 {'id': 'GO_0000002',
  'props': {'name': 'obsolete mitochondrial genome maintenance',
   'definition': 'OBSOLETE. The maintenance of the structure and integrity of the mitochondrial genome; includes replication and segregation of the mitochondrial chromosome.',
   'definition_xref': ['GOC:ai', 'GOC:vw'],
   'hasOBONamespace': 'biological_process',
   'deprecated': True}},
 {'id': 'GO_0000003',
  'props': {'name': 'obsolete reproduction',
   'definition': 'OBSOLETE. The production of new individuals that contain some portion of geneti

In [35]:
for node in graph['nodes']:
    if node['id'].split("/")[-1].startswith('BFO'):
        print(node)

{'id': 'http://purl.obolibrary.org/obo/BFO_0000050', 'lbl': 'part of', 'type': 'PROPERTY', 'propertyType': 'OBJECT', 'meta': {'xrefs': [{'val': 'BFO:0000050'}], 'basicPropertyValues': [{'pred': 'http://www.geneontology.org/formats/oboInOwl#hasOBONamespace', 'val': 'external'}, {'pred': 'http://www.geneontology.org/formats/oboInOwl#shorthand', 'val': 'part_of'}]}}


In [38]:
rare_annot = {e["pred"].split("/")[-1] for e in graph['edges'] if e['pred'] != 'is_a'}
print(rare_annot)


for node in graph['nodes']:
    if node['id'].split('/')[-1] in rare_annot:
        print(node)

{'BFO_0000050', 'RO_0002213', 'RO_0002211', 'subPropertyOf', 'RO_0002212'}
{'id': 'http://purl.obolibrary.org/obo/BFO_0000050', 'lbl': 'part of', 'type': 'PROPERTY', 'propertyType': 'OBJECT', 'meta': {'xrefs': [{'val': 'BFO:0000050'}], 'basicPropertyValues': [{'pred': 'http://www.geneontology.org/formats/oboInOwl#hasOBONamespace', 'val': 'external'}, {'pred': 'http://www.geneontology.org/formats/oboInOwl#shorthand', 'val': 'part_of'}]}}
{'id': 'http://purl.obolibrary.org/obo/RO_0002211', 'lbl': 'regulates', 'type': 'PROPERTY', 'propertyType': 'OBJECT', 'meta': {'xrefs': [{'val': 'RO:0002211'}], 'basicPropertyValues': [{'pred': 'http://www.geneontology.org/formats/oboInOwl#hasOBONamespace', 'val': 'external'}, {'pred': 'http://www.geneontology.org/formats/oboInOwl#shorthand', 'val': 'regulates'}]}}
{'id': 'http://purl.obolibrary.org/obo/RO_0002212', 'lbl': 'negatively regulates', 'type': 'PROPERTY', 'propertyType': 'OBJECT', 'meta': {'xrefs': [{'val': 'RO:0002212'}], 'basicPropertyValue

In [2]:
from local_db.SQLCaller import SQL_ETL

sql_caller = SQL_ETL()
sql_caller.load_annotations_from_diff('CompoundData')

Successfully connected to the database!
Successfully connected to the database!
compound compound_id

            WITH FilteredEntities AS (
                SELECT d.action, d.entity_id, s.entity_type
                FROM diff.entities as d
                INNER JOIN staging.entities as s
                ON d.entity_id = s.entity_id
                WHERE s.entity_type = 'compound'
            )
            SELECT fe.action, fe.entity_id, fe.entity_type, c.*
            FROM FilteredEntities as fe
            LEFT JOIN dbo.CompoundData as c
            ON fe.entity_id = c.compound_id
            


defaultdict(list,
            {('INSERT',
              'C00022',
              'compound'): [{'compound_id': None,
               'compound_name': None,
               'formula': None,
               'compound_synonyms': None,
               'MOL_WEIGHT': None}],
             ('INSERT',
              'C00024',
              'compound'): [{'compound_id': None,
               'compound_name': None,
               'formula': None,
               'compound_synonyms': None,
               'MOL_WEIGHT': None}],
             ('INSERT',
              'C00031',
              'compound'): [{'compound_id': None,
               'compound_name': None,
               'formula': None,
               'compound_synonyms': None,
               'MOL_WEIGHT': None}],
             ('INSERT',
              'C00033',
              'compound'): [{'compound_id': None,
               'compound_name': None,
               'formula': None,
               'compound_synonyms': None,
               'MOL_WEIGHT': No

In [ ]:
from database_callers.SQLCaller import SQL_ETL
from KEGG.KEGGCaller import KEGGCaller

sql_etl = SQL_ETL()
kegg_caller = KEGGCaller()

sql_etl.sql_state.drop_staging_tables()
sql_etl.sql_state.stage_tables()



def parse_kegg_kgml(kegg_caller: KEGGCaller, pathway_id):

    xml = kegg_caller.fetch_pathway_kgml(pathway_id)
    entry_map = kegg_caller.parse_kgml_to_entry_map(xml)

    return kegg_caller.parse_map_sql(entry_map, pathway_id)

def check_kgml_structure(kegg_caller: KEGGCaller, sql_etl: SQL_ETL, pathway_id: str) -> bool:
    
    modified_any = False

    entity_list, intrx_list, rx_list, rx_pt_list = parse_kegg_kgml(kegg_caller, pathway_id)

    diff_ent = sql_etl.stage_and_upsert(target_table='entities', data=entity_list)
    diff_int = sql_etl.stage_and_upsert(target_table='interactions', data=intrx_list)

    if diff_ent or diff_int:
        modified_any = True

    return modified_any




all_data = check_kgml_structure(kegg_caller, sql_etl, pathway_id='hsa00010')


python-dotenv could not parse statement starting at line 2


Successfully connected to the database!
Successfully connected to the database!
Dropped staging table: staging.PathwayIds
Dropped staging table: staging.entities
Dropped staging table: staging.interactions
Cleared staging table: kegg_class
Cleared staging table: PathwayData
Dropped staging table: staging.CompoundData
Dropped staging table: staging.OrthoData
Dropped staging table: staging.reaction_participants
Dropped staging table: staging.reactions
Dropped staging table: staging.GeneData
Table staging.PathwayIds created successfully.
Table staging.entities created successfully.
Table staging.interactions created successfully.
Table staging.CompoundData created successfully.
Table staging.OrthoData created successfully.
Table staging.reaction_participants created successfully.
Table staging.reactions created successfully.
Table staging.GeneData created successfully.

                INSERT INTO staging.entities (entity_id, entity_type)
                SELECT ?, ?
                WHERE 

In [ ]:
from database_callers.SQLCaller import SQLStateManager

state_manager = SQLStateManager()

python-dotenv could not parse statement starting at line 2


Successfully connected to the database!


In [2]:
state_manager.stage_tables()

Table staging.PathwayIds created successfully.
Table staging.entities created successfully.
Table staging.interactions created successfully.
Table staging.CompoundData created successfully.
Table staging.OrthoData created successfully.
Table staging.reaction_participants created successfully.
Table staging.reactions created successfully.
Table staging.GeneData created successfully.


In [3]:
state_manager.drop_staging_tables()

Dropped staging table: staging.PathwayIds
Dropped staging table: staging.entities
Dropped staging table: staging.interactions
Cleared staging table: kegg_class
Cleared staging table: PathwayData
Dropped staging table: staging.CompoundData
Dropped staging table: staging.OrthoData
Dropped staging table: staging.reaction_participants
Dropped staging table: staging.reactions
Dropped staging table: staging.GeneData
